Import packages and functions

In [1]:
import os
import datetime
import torch
import torch.nn as nn
import numpy as np

from MuSt3Net.convolutional_network import CompletionN
from MuSt3Net.normalization_functions import Normalization, Denormalization
from MuSt3Net.training_testing_functions import (
    training_1p, testing_1p,
    training_2p, testing_2p_ensemble
)

from data_preprocessing.get_dataset import *
from data_preprocessing.generation_training_dataset import generate_dataset_phase_2_saving

from utils.utils_general import *
from utils.utils_training import (
    prepare_paths, reload_paths_1p, prepare_paths_2_ensemble,
    generate_training_dataset_1, split_train_test_data,
    load_land_sea_masks, load_old_total_tensor,
    re_load_tensors, recreate_train_test_datasets,
    re_load_transp_lat_coordinates,
    compute_ensemble_mean, compute_ensemble_std,
    compute_3D_ensemble_mean_std
)

from utils.utils_dataset_generation import write_list, read_list, ensure_dir, write_phase2_indexes
from plots.plot_MuSt3Net_output import *

Set the configuration, and select the device for the training 

In [2]:
PATH_JOB = ""  
RUN_TIMESTAMP = str(datetime.datetime.utcnow())
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

/scratch_local/ipykernel_1998000/2060105193.py:2: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  RUN_TIMESTAMP = str(datetime.datetime.utcnow())


Phase 1 - Paths Initialization:   
Create the paths where trainign data, land sea masks and model weights for Phase 1 will be saved. 


In [3]:
paths = prepare_paths(RUN_TIMESTAMP, "P_l", 200, 0, 0.001)
(PATH_JOB, path_results, path_mean_std, path_land_sea_masks, path_configuration, path_lr, path_losses, path_model, path_plots) = paths
job_log = open(os.path.join(PATH_JOB, "file_job_dev.txt"), "a")
job_log.write(f"[first_run_id]: {0}\n")
job_log.close()

Phase 1 – Dataset Loading & Normalization:   
Load data from numerical model and create the input dataset concatenating physical variables and selecting sparse measures of chlorophyll. Normalize the data. 

In [ ]:
n_dupl_per_week = 40  
total_dataset, lat_coords, ywd_indexes = generate_training_dataset_1(
    tensors_directory="dataset_training/total_dataset",
    biogeoch_var="P_l",
    years=[2019],
    n=400,
    n_dupl_per_week=n_dupl_per_week
)
write_list(ywd_indexes, f"{path_lr}/ywd_indexes.txt")
total_dataset_norm, _, _ = Normalization(total_dataset, "1p", path_results)
del total_dataset
(train_ds, val_ds, test_ds, idx_train, idx_val, idx_test) = split_train_test_data(total_dataset_norm)
write_list(idx_train, f"{path_lr}/index_training.txt")
write_list(idx_val, f"{path_lr}/index_internal_testing.txt")
write_list(idx_test, f"{path_lr}/index_external_testing.txt")
land_sea_masks = load_land_sea_masks("dataset_training/land_sea_masks/")

Phase 1 – Training / Testing:   
The function "training_1p" trains the CNN, while "testing_1p" validates the model on unseen data by reproducing 2D maps at different depth layers as well as individual profiles.

In [ ]:
mean_tensor = torch.unsqueeze(torch.load(f"{path_mean_std}/mean_tensor.pt")[:, 6], 1).to(DEVICE)
std_tensor = torch.unsqueeze(torch.load(f"{path_mean_std}/std_tensor.pt")[:, 6], 1).to(DEVICE)
exp_weights = torch.ones([1, 1, d - 2, h, w - 2])
f, f_test = open(path_losses + "/train_loss.txt", "w+"), open(path_losses + "/test_loss.txt", "w+")
training_1p(
    n_epochs_1p=200,
    snaperiod=50,
    l_r=0.001,
    years_week_dupl_indexes=ywd_indexes,
    my_mean_tensor=mean_tensor,
    my_std_tensor=std_tensor,
    train_dataset=train_ds,
    internal_test_dataset=val_ds,
    index_training=idx_train,
    index_internal_test=idx_val,
    land_sea_masks=land_sea_masks,
    exp_weights=exp_weights,
    f=f, 
    f_test=f_test,
    f_job_dev=job_log, 
    losses_1p=[], 
    train_losses_1p=[],
    test_losses_1p=[],
    model_save_path=path_results,
    path_model=path_model,
    path_losses=path_losses,
    path_lr=path_lr,
    transposed_lat_coordinates=lat_coords
)
model = CompletionN()
checkpoint = torch.load(f"{path_results}/model_checkpoint.pth")
model.load_state_dict(checkpoint["model_state_dict"])
testing_1p(
    biogeoch_var="P_l",
    path_plots=path_plots,
    years_week_dupl_indexes=ywd_indexes,
    model_1p=model,
    external_test_dataset=test_ds,
    index_external_testing=idx_test,
    land_sea_masks=land_sea_masks,
    transposed_lat_coordinates=lat_coords,
    my_mean_tensor=mean_tensor,
    my_std_tensor=std_tensor
)

Phase 2 - Paths Initialization:   
Create the paths where input data, land sea masks and model weights for Phase 2 will be saved. 

In [ ]:
N_ENSEMBLE = 10
(path_results, path_mean_std, path_land_sea_masks,
    path_configuration, path_lr, path_losses,
    path_model, path_plots) = reload_paths_1p(
    PATH_JOB, 'P_l', 200, 0, 0.001)
land_sea_masks = load_land_sea_masks("dataset_training/land_sea_masks/")
(path_results_2, path_configuration_2, path_mean_std_2,
    path_lr_2, paths_ensemble_models) = prepare_paths_2_ensemble(
    PATH_JOB, "P_l", 20, 0, 0.001, N_ENSEMBLE)
n_epochs_2p = 20
snaperiod_2p = 5
l_r_2p = 0.001

Phase 2 - Training / Ensemble Testing:  
The function "training_2p" trains the CNN for the Phase 2, reloading and starting from the CNN weights of Phase 1, while "testing_2p_ensemble" validates the ensemble models on unseen data by reproducing 2D maps at different depth layers as well as individual profiles.  

In [ ]:
for i_ens in range(N_ENSEMBLE):
    (list_year_week_indexes,old_float_total_dataset,list_float_profiles_coordinates,
    sampled_list_float_profile_coordinates,index_training_2,index_internal_testing_2,
    index_external_testing_2,train_dataset_2,internal_test_dataset_2,
    test_dataset_2) = generate_dataset_phase_2_saving(
    "P_l", path_results_2, [2019],
    "dataset_training/float", land_sea_masks
    )
    write_phase2_indexes(
        path_lr_2,
        list_year_week_indexes,
        index_training_2,
        index_internal_testing_2,
        index_external_testing_2
    )
    path_ens = paths_ensemble_models[i_ens]
    write_phase2_indexes(
        path_ens,
        list_year_week_indexes,
        index_training_2,
        index_internal_testing_2,
        index_external_testing_2
    )
    path_losses_2 = os.path.join(path_ens, "losses")
    path_model_2 = os.path.join(path_ens, "partial_models")
    path_plots_2 = os.path.join(path_ens, "plots")
    for p in (path_losses_2, path_model_2, path_plots_2):
        ensure_dir(p)

    f_2 = open(os.path.join(path_losses_2, "train_losses_2p.txt"), "w+")
    f_2_test = open(os.path.join(path_losses_2, "test_losses_2p.txt"), "w+")
    my_mean_tensor_2p = torch.unsqueeze(torch.load(os.path.join(path_mean_std_2, "mean_tensor.pt"))[:, 6], 1).to(DEVICE)
    my_std_tensor_2p = torch.unsqueeze(torch.load(os.path.join(path_mean_std_2, "std_tensor.pt"))[:, 6], 1).to(DEVICE)
    exp_weights = torch.ones([1, 1, d-2, h, w-2])
    training_2p(
        n_epochs_2p, snaperiod_2p, l_r_2p,
        my_mean_tensor_2p, my_std_tensor_2p,
        train_dataset_2, internal_test_dataset_2,
        index_training_2, index_internal_testing_2,
        land_sea_masks, exp_weights,
        old_float_total_dataset,
        sampled_list_float_profile_coordinates,
        f_2, f_2_test,
        [], [], [],
        path_results, path_results_2,
        path_model_2, path_losses_2
    )
    f_2.close()
    f_2_test.close()
    src = os.path.join(path_results_2, "model_checkpoint_2.pth")
    dst = os.path.join(path_ens, f"model_checkpoint_2_ens_{i_ens}.pth")
    with open(src, "rb") as fsrc, open(dst, "wb") as fdst:
        fdst.write(fsrc.read())
    os.remove(src)

    model_1p = CompletionN()
    ckpt_1p = torch.load(os.path.join(path_results, "model_checkpoint.pth"))
    model_1p.load_state_dict(ckpt_1p["model_state_dict"])
    model_1p.eval()

    model_2p = CompletionN()
    ckpt_2p = torch.load(dst)
    model_2p.load_state_dict(ckpt_2p["model_state_dict"])
    model_2p.eval()

    biogeoch_total_dataset = [
        torch.unsqueeze(load_old_total_tensor("dataset_training/old_total_dataset/",i_test, list_year_week_indexes)[:, -1], 1)
        for i_test in index_external_testing_2
    ]
    testing_2p_ensemble(
        "P_l", path_plots_2, list_year_week_indexes,
        biogeoch_total_dataset, old_float_total_dataset,
        model_1p, model_2p,
        test_dataset_2, index_external_testing_2,
        land_sea_masks,
        list_float_profiles_coordinates,
        sampled_list_float_profile_coordinates,
        my_mean_tensor_2p, my_std_tensor_2p,
        exp_weights, path_losses_2
    )

Reload Phase-1 model ONCE for ensemble evaluation

In [ ]:
model_1p = CompletionN().to(DEVICE)
ckpt_1p = torch.load(os.path.join(path_results, "model_checkpoint.pth"))
model_1p.load_state_dict(ckpt_1p["model_state_dict"])
model_1p.eval()

Build biogeochemical reference dataset, useful for profiles comparison with BGC-Argo floats

In [ ]:
biogeoch_total_dataset = [
    torch.unsqueeze(
        load_old_total_tensor(
            "dataset_training/old_total_dataset/",
            i_test,
            list_year_week_indexes
        )[:, -1], 1) for i_test in index_external_testing_2]

Compute the ensemble model (constructed by averaging the outputs of multiple trained Phase 2 CNN instances).    
Compute Ensemble Mean and Standard deviation, and plots the resulted maps and profiles. 

In [ ]:
models_list = []
for i_ens in range(N_ENSEMBLE):
    model = CompletionN().to(DEVICE)
    ckpt = torch.load(
        os.path.join(
            paths_ensemble_models[i_ens],
            f"model_checkpoint_2_ens_{i_ens}.pth"
        )
    )
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()
    models_list.append(model)

EPS = 1e-8
path_ensemble_plots = os.path.join(path_lr_2, "plots_ensemble")
ensure_dir(path_ensemble_plots)

for i_test, sample in enumerate(test_dataset_2):
    year, week = list_year_week_indexes[index_external_testing_2[i_test]]
    base = os.path.join(
        path_ensemble_plots,
        f"test_data_year_{year}_week_{week}"
    )
    mean_p = os.path.join(base, "mean")
    std_p = os.path.join(base, "std")
    prof_p = os.path.join(base, "profiles")
    for p in (mean_p, std_p, prof_p):
        ensure_dir(p)

    ens_mean, ens_std = compute_3D_ensemble_mean_std(sample, models_list, path_mean_std_2)
    plot_NN_maps(ens_mean, land_sea_masks, "P_l", mean_p)
    plot_NN_maps_std_percentage(ens_std / (ens_mean + EPS) * 100,land_sea_masks, "P_l", std_p)

    with torch.no_grad():
        nn_1p_out = model_1p(sample.to(DEVICE))
        nn_1p_out = Denormalization(nn_1p_out, my_mean_tensor_2p, my_std_tensor_2p)
    tensor_output_float = torch.unsqueeze(old_float_total_dataset[index_external_testing_2[i_test]][:, 6], 1)
    tensor_output_num_model = biogeoch_total_dataset[i_test][:, :, :-1, :, 1:-1]
    comparison_profiles_1_2_phases(tensor_output_float,ens_mean,tensor_output_num_model,nn_1p_out,"P_l",prof_p) 
